# Entropy vs. Generative Perplexity Comparison

This notebook collects and plots the results from evaluation sweeps across different values of `p_nucleus` and `temperature` at a fixed step limit (e.g., $T = 256$).

In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
def extract_sweep_data(experiments_dir, algo, t_val, param_name, values, keys_to_extract=["generative_ppl", "entropy"]):
    """
    Extracts generation metrics across different parameter sweep values for a fixed T.
    """
    results = []
    base_dir = Path(experiments_dir)
    
    for val in values:
        row_data = {
            "algo": algo,
            "T": t_val,
            param_name: val
        }
        # Initialize metrics with None
        for key in keys_to_extract:
            row_data[key] = None
            
        filename = f"{algo}_T-{t_val}_{param_name}-{val}.json"
        file_path = base_dir / filename
        
        if file_path.exists():
            try: 
                with file_path.open('r') as f:
                    data = json.load(f)
                for key in keys_to_extract:
                    row_data[key] = data.get(key)
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
                
        results.append(row_data)
        
    return pd.DataFrame(results)

In [ ]:
# Configure sweep directories and values
SWEEPS_DIR = "../outputs/sweeps"
TARGET_T = 256
P_NUCLEUS_VALUES = [0.8, 0.825, 0.85, 0.875, 0.9, 0.925, 0.95, 0.975, 1.0]
TEMPERATURE_VALUES = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]

algos_to_plot = ["mdlm", "sedd", "duo", "smflm", "flm"]
all_data = []

for algo in algos_to_plot:
    if algo == "flm":
        df = extract_sweep_data(SWEEPS_DIR, algo, TARGET_T, "temperature", TEMPERATURE_VALUES)
    else:
        df = extract_sweep_data(SWEEPS_DIR, algo, TARGET_T, "p_nucleus", P_NUCLEUS_VALUES)
    all_data.append(df)

df_all = pd.concat(all_data, ignore_index=True)
df_all.dropna(subset=["generative_ppl", "entropy"], inplace=True)
df_all

In [ ]:
# Plot the tradeoff curves
plt.figure(figsize=(10, 6))

for algo, group in df_all.groupby("algo"):
    # Sort by entropy for a continuous line
    group_sorted = group.sort_values(by="entropy")
    
    plt.plot(
        group_sorted["entropy"], 
        group_sorted["generative_ppl"], 
        marker='o', 
        linestyle='-', 
        linewidth=2,
        label=algo.upper()
    )

plt.title(f"Entropy vs. Generative Perplexity (T = {TARGET_T})", fontsize=14, fontweight='bold')
plt.xlabel("Sample Entropy", fontsize=12)
plt.ylabel("Generative Perplexity", fontsize=12)
plt.grid(True, which="both", linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

output_file = "entropy_vs_perplexity_curve.png"
plt.savefig(output_file, dpi=300)
print(f"Plot saved successfully to {output_file}")
plt.show()